# Plot trajectories

Examples for the new `functions/sc_pos` tooling: time-series, 3D orbits, and animated lon/lat overlays.


In [ ]:
import os, sys, importlib

USER_REPO_ROOT = r"C:\Users\nokni\work\MHDTurbPy"  # set once here
SC_POS_PATH = os.path.join(USER_REPO_ROOT, "functions", "sc_pos")
os.chdir(SC_POS_PATH)
sys.path.insert(0, os.getcwd())

import helpers, horizons_sun_lonlat
importlib.reload(helpers)
importlib.reload(horizons_sun_lonlat)

from interactive_orbits_timeseries_plus3d import build_timeseries_figure, build_3d_figure

targets = ["PSP"]  # add more: "ACE", "Wind", "IMAP", "SOLAR-1"

fig_ts = build_timeseries_figure(
    targets=targets,
    start="2020-02-01T00:00:00",
    stop="2026-02-10T00:00:00",
    step="6h",
    rss_rsun=2.5,
    omega_deg_per_day=14.1844,
    width=1800,
    height=1100,
)
fig_ts.show()

# NOTE: current function signature uses frame3d/vsw1_kms/vsw2_kms.
fig_3d = build_3d_figure(
    targets=targets,
    start="2019-10-01T00:00:00",
    stop="2021-02-10T00:00:00",
    step="6h",
    frame3d="HCI",
    width=1800,
    height=900,
)
fig_3d.show()

#!python interactive_orbits_timeseries_plus3d.py --start 2026-02-01T00:00:00 --stop 2026-02-10T00:00:00 --step 6h --targets ACE Wind IMAP SOLAR-1 --html out_orbits/interactive_orbits_plus3d.html --frame3d HCI



## Animated 3D lon/lat panel
This example adds a slider and a side annotation with lon/lat/r for each spacecraft over time.


In [ ]:
from __future__ import annotations

import argparse
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from horizons_sun_lonlat import get_lonlat_xyz_timeseries


DEFAULT_ALIASES = {
    "SOLAR-1": ["SWFO-L1", "Space weather Observations at L1 to Advance Readiness - 1"],
    "SWFO-L1": ["SOLAR-1", "Space weather Observations at L1 to Advance Readiness - 1"],
    "IMAP": ["Interstellar Mapping and Acceleration Probe"],
    "ACE": ["Advanced Composition Explorer"],
    "WIND": ["Wind"],
}


def _au_to_r(x_au, y_au, z_au):
    x = np.asarray(x_au, dtype=float)
    y = np.asarray(y_au, dtype=float)
    z = np.asarray(z_au, dtype=float)
    return np.sqrt(x * x + y * y + z * z)


def build_figure(series: Dict[str, pd.DataFrame], title: str) -> go.Figure:
    names = list(series.keys())
    t = series[names[0]].index
    nT = len(t)

    fig = go.Figure()

    for name in names:
        df = series[name]
        fig.add_trace(go.Scatter3d(x=df["hee_x_au"], y=df["hee_y_au"], z=df["hee_z_au"], mode="lines", name=f"{name} (traj)"))
        fig.add_trace(go.Scatter3d(x=[df["hee_x_au"].iloc[0]], y=[df["hee_y_au"].iloc[0]], z=[df["hee_z_au"].iloc[0]], mode="markers", name=f"{name} (t)"))

    fig.add_trace(go.Scatter3d(x=[0.0], y=[0.0], z=[0.0], mode="markers", name="Sun"))

    frames = []
    marker_trace_indices = [2 * i + 1 for i, _ in enumerate(names)]

    for k in range(nT):
        data_updates = []
        for name in names:
            df = series[name]
            data_updates.append(go.Scatter3d(x=[df["hee_x_au"].iloc[k]], y=[df["hee_y_au"].iloc[k]], z=[df["hee_z_au"].iloc[k]], mode="markers"))

        lines = [f"<b>{t[k].strftime('%Y-%m-%d %H:%M UTC')}</b>"]
        for name in names:
            df = series[name]
            lines.append(f"{name}: lon={df['hgs_lon_deg'].iloc[k]:8.3f} deg, lat={df['hgs_lat_deg'].iloc[k]:8.3f} deg, r={_au_to_r(df['hee_x_au'].iloc[k], df['hee_y_au'].iloc[k], df['hee_z_au'].iloc[k]):.6f} AU")

        frames.append(go.Frame(name=str(k), data=data_updates, traces=marker_trace_indices, layout=go.Layout(annotations=[go.layout.Annotation(x=0.02, y=0.98, xref="paper", yref="paper", showarrow=False, align="left", text="<br>".join(lines), bordercolor="black", borderwidth=1, bgcolor="white", opacity=0.9)])))

    fig.frames = frames

    steps = []
    for k in range(nT):
        steps.append(dict(method="animate", args=[[str(k)], dict(frame=dict(duration=0, redraw=True), mode="immediate", transition=dict(duration=0))], label=str(k)))

    fig.update_layout(
        title=title,
        scene=dict(xaxis_title="x [AU]", yaxis_title="y [AU]", zaxis_title="z [AU]", aspectmode="data"),
        updatemenus=[dict(type="buttons", showactive=False, x=0.02, y=0.02, xanchor="left", yanchor="bottom", buttons=[dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=60, redraw=True), fromcurrent=True)]), dict(label="Pause", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])],
        sliders=[dict(x=0.15, y=0.02, len=0.8, currentvalue=dict(prefix="t index = "), steps=steps)],
    )
    return fig


def main() -> None:
    p = argparse.ArgumentParser()
    p.add_argument("--start", required=True, help="UTC start, e.g. 2026-01-20T00:00")
    p.add_argument("--stop", required=True, help="UTC stop, e.g. 2026-01-27T00:00")
    p.add_argument("--step", default="1h", help="Horizons step, e.g. 5m, 1h, 6h, 1d")
    p.add_argument("--targets", nargs="+", default=["IMAP", "SOLAR-1", "ACE", "WIND"])
    p.add_argument("--outdir", default="out_orbits_lonlat")
    p.add_argument("--html", default="orbits_lonlat.html")
    args = p.parse_args()

    outdir = Path(args.outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    series: Dict[str, pd.DataFrame] = {}
    for tgt in args.targets:
        aliases = DEFAULT_ALIASES.get(tgt, [])
        try_order = [tgt] + aliases
        last_err = None
        ts = None
        for name_try in try_order:
            try:
                ts = get_lonlat_xyz_timeseries(name_try, args.start, args.stop, args.step, carrington=False)
                break
            except Exception as exc:
                last_err = exc

        if ts is None:
            raise RuntimeError(f"Failed to resolve/load {tgt}. Last error: {last_err}")

        df = ts.df.copy()
        df.to_csv(outdir / f"{tgt.replace('/', '_')}_lonlat.csv")
        series[tgt] = df

    fig = build_figure(series, title="Sun-centered ecliptic trajectories + lon/lat (from JPL Horizons)")
    fig.write_html(str(outdir / args.html), include_plotlyjs="cdn")
    print(f"Wrote: {outdir / args.html}")
    print(f"CSV files: {outdir}")


# Example direct run (uncomment in terminal-style usage):
# main()
